# GPTQ MLP-only notebook — Perplexity + GSM8K + ARC-Challenge

This notebook runs the same execution order as the full-quant notebook, but for **Qwen2-1.5B MLP-only GPTQ quantization**:

1. quantize and save the model artifact
2. evaluate **Perplexity** on WikiText-2
3. evaluate **GSM8K**
4. evaluate **ARC-Challenge**

It saves:
- quantized GPTQ artifact folder
- quantized model zip
- perplexity / GSM8K / ARC-Challenge metrics and predictions
- summary table row
- results-only zip


In [ ]:

# Colab / notebook setup
!pip install -U pip setuptools wheel
!pip install -U gptqmodel
!pip install -q torch transformers accelerate datasets pandas pyarrow safetensors tqdm


  Using cached gptqmodel-5.7.0.tar.gz (668 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached device_smi-0.5.3.tar.gz (18 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 67.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 

KeyboardInterrupt: 

In [ ]:

# Restart once after installation in a fresh runtime, then skip this cell on reruns.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

In [ ]:

from __future__ import annotations

import json
import logging
import math
import os
import re
import time
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import GPTQModel, QuantizeConfig

# ------------------------------
# Logging
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    force=True,
)
logger = logging.getLogger('gptq_mlp_only_quant')

PROJECT_ROOT = Path.cwd().resolve()
VARIANT = 'mlp_only_quant'
RUN_NAME = 'gptq_w4_g128_mlp_only_quant'
BASE_MODEL = 'Qwen/Qwen2-1.5B'

cfg = {
    'base_model': BASE_MODEL,
    'calibration': {
        'dataset': 'wikitext',
        'dataset_name': 'wikitext-2-raw-v1',
        'split': 'train',
        'num_samples': 512,
        'max_length': 2048,
        'seed': 42,
    },
    'quant': {
        'bits': 4,
        'group_size': 128,
        'desc_act': False,
    },
    'eval': {
        'perplexity': {
            'dataset': 'wikitext',
            'dataset_name': 'wikitext-2-raw-v1',
            'split': 'test',
            'max_length': 2048,
            'stride': 1024,
            'max_eval_tokens': 131072,
            'log_every': 25,
        },
        'gsm8k': {
            'num_fewshot': 8,
            'num_samples': 300,
            'max_new_tokens': 256,
            'log_every': 25,
        },
        'arc_challenge': {
            'num_samples': 500,
            'max_new_tokens': 5,
            'log_every': 50,
        },
    },
    'paths': {
        'artifacts_dir': str(PROJECT_ROOT / 'artifacts'),
        'results_dir': str(PROJECT_ROOT / 'results' / RUN_NAME),
        'calibration_dir': str(PROJECT_ROOT / 'calibration_data'),
        'bundles_dir': str(PROJECT_ROOT / 'zip_bundles' / RUN_NAME),
    },
}

PATHS = {k: Path(v) for k, v in cfg['paths'].items()}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PATHS['artifacts_dir'] / 'gptq_w4_g128' / VARIANT
RESULTS_DIR = PATHS['results_dir'] / VARIANT
BUNDLE_DIR = PATHS['bundles_dir'] / VARIANT
CALIB_DIR = PATHS['calibration_dir']
for p in [ARTIFACT_DIR, RESULTS_DIR, BUNDLE_DIR, CALIB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def build_dynamic_config(variant: str) -> dict | None:
    if variant == 'full_quant':
        return None
    if variant == 'attn_only_quant':
        return {r'-:.*mlp\.(gate|up|down)_proj.*': {}}
    if variant == 'mlp_only_quant':
        return {r'-:.*self_attn\.(q|k|v|o)_proj.*': {}}
    raise ValueError(f'Unknown variant: {variant}')

print('Working directory:', PROJECT_ROOT)
print('Artifact dir:', ARTIFACT_DIR)
print('Results dir:', RESULTS_DIR)
print('Bundle dir:', BUNDLE_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ------------------------------
# Small utilities
# ------------------------------
def save_json(obj: Any, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)
    return path


def read_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def zip_directory(source_dir: str | Path, zip_path: str | Path, *, exclude_suffixes=('.lock',), exclude_dirs=('.ipynb_checkpoints',)):
    source_dir = Path(source_dir).resolve()
    zip_path = Path(zip_path).resolve()
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_zip = zip_path.with_suffix(zip_path.suffix + '.tmp')
    if tmp_zip.exists():
        tmp_zip.unlink()
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for path in sorted(source_dir.rglob('*')):
            if path.is_dir():
                continue
            if any(part in exclude_dirs for part in path.parts):
                continue
            if path.resolve() in {zip_path, tmp_zip}:
                continue
            if path.suffix in exclude_suffixes:
                continue
            zf.write(path, arcname=str(path.relative_to(source_dir)))

    tmp_zip.replace(zip_path)
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    logger.info('Created zip: %s (%.2f MB)', zip_path, size_mb)
    return zip_path


def find_gptq_model_basename(model_dir: str | Path) -> str | None:
    model_dir = Path(model_dir)
    candidates = sorted(model_dir.glob('*.safetensors'))
    if not candidates:
        return None
    preferred = [p for p in candidates if 'gptq' in p.name.lower()]
    return (preferred[0] if preferred else candidates[0]).name


def estimate_num_params_from_config(model_dir: str | Path) -> int | None:
    cfg_path = Path(model_dir) / 'config.json'
    if not cfg_path.exists():
        return None
    cfg_json = json.loads(cfg_path.read_text())
    hidden_size = cfg_json.get('hidden_size')
    num_hidden_layers = cfg_json.get('num_hidden_layers')
    intermediate_size = cfg_json.get('intermediate_size')
    vocab_size = cfg_json.get('vocab_size')
    if not all(v is not None for v in [hidden_size, num_hidden_layers, intermediate_size, vocab_size]):
        return None
    rough_params = (
        vocab_size * hidden_size +
        num_hidden_layers * (4 * hidden_size * hidden_size) +
        num_hidden_layers * (3 * hidden_size * intermediate_size)
    )
    return int(rough_params)


def model_size_gb(model_dir: str | Path) -> float:
    files = list(Path(model_dir).glob('*.safetensors'))
    total_bytes = sum(p.stat().st_size for p in files)
    return total_bytes / (1024 ** 3)


def bits_per_param(model_dir: str | Path) -> float | None:
    files = list(Path(model_dir).glob('*.safetensors'))
    if not files:
        return None
    total_bytes = sum(p.stat().st_size for p in files)
    n_params = estimate_num_params_from_config(model_dir)
    if not n_params:
        return None
    return 8.0 * total_bytes / n_params


def get_results_state() -> dict:
    state = read_json(RESULTS_DIR / 'state.json', default={})
    return state or {}


def save_results_state(state: dict):
    save_json(state, RESULTS_DIR / 'state.json')
    return state


def update_summary_table(state: dict):
    size_gb = model_size_gb(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    bpp = bits_per_param(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    peak_candidates = []
    for key in ['perplexity', 'gsm8k', 'arc_challenge']:
        if key in state and isinstance(state[key], dict):
            val = state[key].get('peak_vram_gb')
            if val is not None:
                peak_candidates.append(val)

    summary_row = {
        'variant': VARIANT,
        'compression': f"GPTQ W{cfg['quant']['bits']} g{cfg['quant']['group_size']}",
        'model_size_gb': round(size_gb, 4) if size_gb is not None else None,
        'peak_vram_gb': round(max(peak_candidates), 4) if peak_candidates else None,
        'bits_per_param': round(bpp, 4) if bpp is not None else None,
        'perplexity': state.get('perplexity', {}).get('perplexity'),
        'gsm8k_accuracy': state.get('gsm8k', {}).get('accuracy'),
        'arc_accuracy': state.get('arc_challenge', {}).get('accuracy'),
    }
    save_json(summary_row, RESULTS_DIR / 'summary.json')
    pd.DataFrame([summary_row]).to_csv(RESULTS_DIR / 'summary.csv', index=False)
    return summary_row


def _is_gptq_checkpoint(model_path: str | Path) -> bool:
    cfg_path = Path(model_path) / 'config.json'
    if not cfg_path.exists():
        return False
    cfg_json = json.loads(cfg_path.read_text())
    return cfg_json.get('quantization_config', {}).get('quant_method') == 'gptq'


def load_quantized_model_and_tokenizer(model_path: str | Path):
    model_path = Path(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    if not _is_gptq_checkpoint(model_path):
        raise ValueError(f'{model_path} is not a GPTQ checkpoint.')

    model_basename = find_gptq_model_basename(model_path)
    if model_basename is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {model_path}')

    model = GPTQModel.load(
        str(model_path),
        device_map='auto',
        trust_remote_code=True,
        model_basename=model_basename,
    )
    model.eval()
    return model, tokenizer


def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        first_device = next(iter(model.hf_device_map.values()))
        return torch.device(first_device)
    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def generate_text(model, tokenizer, prompt: str, *, max_new_tokens: int):
    device = get_model_device(model)
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=min(getattr(tokenizer, 'model_max_length', 4096), 4096),
        padding=False,
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    gen_ids = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


ANSWER_RE = re.compile(r'####\s*(-?[\d,]+\.?\d*)')


def extract_numeric_answer(text: str):
    m = ANSWER_RE.search(text)
    if m:
        try:
            return float(m.group(1).replace(',', ''))
        except ValueError:
            return None
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None


def format_seconds(sec: float) -> str:
    sec = int(max(sec, 0))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f'{h:d}:{m:02d}:{s:02d}'
    return f'{m:d}:{s:02d}'


def log_progress(prefix: str, idx: int, total: int, start_time: float, correct: int):
    elapsed = time.time() - start_time
    avg = elapsed / max(idx, 1)
    eta = avg * max(total - idx, 0)
    logger.info(
        '%s | %d/%d | acc=%.4f | elapsed=%s | eta=%s | %.2fs/item',
        prefix,
        idx,
        total,
        correct / max(idx, 1),
        format_seconds(elapsed),
        format_seconds(eta),
        avg,
    )


Working directory: /content
Artifact dir: /content/artifacts/gptq_w4_g128/mlp_only_quant
Results dir: /content/results/gptq_w4_g128_mlp_only_quant/mlp_only_quant
Bundle dir: /content/zip_bundles/gptq_w4_g128_mlp_only_quant/mlp_only_quant
CUDA available: True
GPU: NVIDIA L4


In [ ]:
import json
import logging
import math
import os
import re
import time
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import GPTQModel, QuantizeConfig

# ------------------------------
# Logging
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    force=True,
)
logger = logging.getLogger('gptq_mlp_only_quant')

PROJECT_ROOT = Path.cwd().resolve()
VARIANT = 'mlp_only_quant'
RUN_NAME = 'gptq_w4_g128_mlp_only_quant'
BASE_MODEL = 'Qwen/Qwen2-1.5B'

cfg = {
    'base_model': BASE_MODEL,
    'calibration': {
        'dataset': 'wikitext',
        'dataset_name': 'wikitext-2-raw-v1',
        'split': 'train',
        'num_samples': 512,
        'max_length': 2048,
        'seed': 42,
    },
    'quant': {
        'bits': 4,
        'group_size': 128,
        'desc_act': False,
    },
    'eval': {
        'perplexity': {
            'dataset': 'wikitext',
            'dataset_name': 'wikitext-2-raw-v1',
            'split': 'test',
            'max_length': 2048,
            'stride': 1024,
            'max_eval_tokens': 131072,
            'log_every': 25,
        },
        'gsm8k': {
            'num_fewshot': 8,
            'num_samples': 300,
            'max_new_tokens': 256,
            'log_every': 25,
        },
        'arc_challenge': {
            'num_samples': 500,
            'max_new_tokens': 5,
            'log_every': 50,
        },
    },
    'paths': {
        'artifacts_dir': str(PROJECT_ROOT / 'artifacts'),
        'results_dir': str(PROJECT_ROOT / 'results' / RUN_NAME),
        'calibration_dir': str(PROJECT_ROOT / 'calibration_data'),
        'bundles_dir': str(PROJECT_ROOT / 'zip_bundles' / RUN_NAME),
    },
}

PATHS = {k: Path(v) for k, v in cfg['paths'].items()}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PATHS['artifacts_dir'] / 'gptq_w4_g128' / VARIANT
RESULTS_DIR = PATHS['results_dir'] / VARIANT
BUNDLE_DIR = PATHS['bundles_dir'] / VARIANT
CALIB_DIR = PATHS['calibration_dir']
for p in [ARTIFACT_DIR, RESULTS_DIR, BUNDLE_DIR, CALIB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def build_dynamic_config(variant: str) -> dict | None:
    if variant == 'full_quant':
        return None
    if variant == 'attn_only_quant':
        return {r'-:.*mlp\.(gate|up|down)_proj.*': {}}
    if variant == 'mlp_only_quant':
        return {r'-:.*self_attn\.(q|k|v|o)_proj.*': {}}
    raise ValueError(f'Unknown variant: {variant}')


print('Working directory:', PROJECT_ROOT)
print('Artifact dir:', ARTIFACT_DIR)
print('Results dir:', RESULTS_DIR)
print('Bundle dir:', BUNDLE_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


# ------------------------------
# Small utilities
# ------------------------------
def save_json(obj: Any, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)
    return path


def read_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def zip_directory(source_dir: str | Path, zip_path: str | Path, *, exclude_suffixes=('.lock',), exclude_dirs=('.ipynb_checkpoints',)):
    source_dir = Path(source_dir).resolve()
    zip_path = Path(zip_path).resolve()
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_zip = zip_path.with_suffix(zip_path.suffix + '.tmp')
    if tmp_zip.exists():
        tmp_zip.unlink()
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for path in sorted(source_dir.rglob('*')):
            if path.is_dir():
                continue
            if any(part in exclude_dirs for part in path.parts):
                continue
            if path.resolve() in {zip_path, tmp_zip}:
                continue
            if path.suffix in exclude_suffixes:
                continue
            zf.write(path, arcname=str(path.relative_to(source_dir)))

    tmp_zip.replace(zip_path)
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    logger.info('Created zip: %s (%.2f MB)', zip_path, size_mb)
    return zip_path


def find_gptq_model_basename(model_dir: str | Path) -> str | None:
    model_dir = Path(model_dir)
    candidates = sorted(model_dir.glob('*.safetensors'))
    if not candidates:
        return None
    preferred = [p for p in candidates if 'gptq' in p.name.lower()]
    return (preferred[0] if preferred else candidates[0]).name


def estimate_num_params_from_config(model_dir: str | Path) -> int | None:
    cfg_path = Path(model_dir) / 'config.json'
    if not cfg_path.exists():
        return None
    cfg_json = json.loads(cfg_path.read_text())
    hidden_size = cfg_json.get('hidden_size')
    num_hidden_layers = cfg_json.get('num_hidden_layers')
    intermediate_size = cfg_json.get('intermediate_size')
    vocab_size = cfg_json.get('vocab_size')
    if not all(v is not None for v in [hidden_size, num_hidden_layers, intermediate_size, vocab_size]):
        return None
    rough_params = (
        vocab_size * hidden_size +
        num_hidden_layers * (4 * hidden_size * hidden_size) +
        num_hidden_layers * (3 * hidden_size * intermediate_size)
    )
    return int(rough_params)


def model_size_gb(model_dir: str | Path) -> float:
    files = list(Path(model_dir).glob('*.safetensors'))
    total_bytes = sum(p.stat().st_size for p in files)
    return total_bytes / (1024 ** 3)


def bits_per_param(model_dir: str | Path) -> float | None:
    files = list(Path(model_dir).glob('*.safetensors'))
    if not files:
        return None
    total_bytes = sum(p.stat().st_size for p in files)
    n_params = estimate_num_params_from_config(model_dir)
    if not n_params:
        return None
    return 8.0 * total_bytes / n_params


def get_results_state() -> dict:
    state = read_json(RESULTS_DIR / 'state.json', default={})
    return state or {}


def save_results_state(state: dict):
    save_json(state, RESULTS_DIR / 'state.json')
    return state


def update_summary_table(state: dict):
    size_gb = model_size_gb(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    bpp = bits_per_param(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    peak_candidates = []
    for key in ['perplexity', 'gsm8k', 'arc_challenge']:
        if key in state and isinstance(state[key], dict):
            val = state[key].get('peak_vram_gb')
            if val is not None:
                peak_candidates.append(val)

    summary_row = {
        'variant': VARIANT,
        'compression': f"GPTQ W{cfg['quant']['bits']} g{cfg['quant']['group_size']}",
        'model_size_gb': round(size_gb, 4) if size_gb is not None else None,
        'peak_vram_gb': round(max(peak_candidates), 4) if peak_candidates else None,
        'bits_per_param': round(bpp, 4) if bpp is not None else None,
        'perplexity': state.get('perplexity', {}).get('perplexity'),
        'gsm8k_accuracy': state.get('gsm8k', {}).get('accuracy'),
        'arc_accuracy': state.get('arc_challenge', {}).get('accuracy'),
    }
    save_json(summary_row, RESULTS_DIR / 'summary.json')
    pd.DataFrame([summary_row]).to_csv(RESULTS_DIR / 'summary.csv', index=False)
    return summary_row


def _is_gptq_checkpoint(model_path: str | Path) -> bool:
    cfg_path = Path(model_path) / 'config.json'
    if not cfg_path.exists():
        return False
    cfg_json = json.loads(cfg_path.read_text())
    return cfg_json.get('quantization_config', {}).get('quant_method') == 'gptq'


def load_quantized_model_and_tokenizer(model_path: str | Path):
    model_path = Path(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    if not _is_gptq_checkpoint(model_path):
        raise ValueError(f'{model_path} is not a GPTQ checkpoint.')

    model_basename = find_gptq_model_basename(model_path)
    if model_basename is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {model_path}')

    model = GPTQModel.load(
        str(model_path),
        device_map='auto',
        trust_remote_code=True,
        model_basename=model_basename,
    )
    model.eval()
    return model, tokenizer


def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        first_device = next(iter(model.hf_device_map.values()))
        return torch.device(first_device)
    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def generate_text(model, tokenizer, prompt: str, *, max_new_tokens: int):
    device = get_model_device(model)
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=min(getattr(tokenizer, 'model_max_length', 4096), 4096),
        padding=False,
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    gen_ids = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


ANSWER_RE = re.compile(r'####\s*(-?[\d,]+\.?\d*)')


def extract_numeric_answer(text: str):
    m = ANSWER_RE.search(text)
    if m:
        try:
            return float(m.group(1).replace(',', ''))
        except ValueError:
            return None
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None


def format_seconds(sec: float) -> str:
    sec = int(max(sec, 0))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f'{h:d}:{m:02d}:{s:02d}'
    return f'{m:d}:{s:02d}'


def log_progress(prefix: str, idx: int, total: int, start_time: float, correct: int):
    elapsed = time.time() - start_time
    avg = elapsed / max(idx, 1)
    eta = avg * max(total - idx, 0)
    logger.info(
        '%s | %d/%d | acc=%.4f | elapsed=%s | eta=%s | %.2fs/item',
        prefix,
        idx,
        total,
        correct / max(idx, 1),
        format_seconds(elapsed),
        format_seconds(eta),
        format_seconds(eta),
        avg,
    )


# Quantization helpers

def get_calibration_data(cfg, tokenizer=None):
    cal_cfg = cfg['calibration']
    cache_path = CALIB_DIR / f"calibration_{cal_cfg['num_samples']}_{cal_cfg['max_length']}.pt"
    if cache_path.exists():
        logger.info('Using cached calibration samples: %s', cache_path)
        return torch.load(cache_path, weights_only=False)

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

    logger.info('Loading calibration dataset: %s / %s / %s', cal_cfg['dataset'], cal_cfg['dataset_name'], cal_cfg['split'])
    ds = load_dataset(cal_cfg['dataset'], cal_cfg['dataset_name'], split=cal_cfg['split'])
    all_text = '\n\n'.join(t for t in ds['text'] if t and t.strip())
    encoded = tokenizer(all_text, return_tensors='pt')
    total_tokens = encoded.input_ids.shape[1]
    max_len = cal_cfg['max_length']

    if total_tokens <= max_len:
        raise ValueError(f'Calibration corpus too short: total_tokens={total_tokens}, max_length={max_len}')

    rng = torch.Generator().manual_seed(cal_cfg['seed'])
    starts = torch.randint(0, total_tokens - max_len, (cal_cfg['num_samples'],), generator=rng)
    samples = []
    for start in tqdm(starts.tolist(), desc='Preparing calibration windows', unit='sample'):
        end = start + max_len
        input_ids = encoded.input_ids[:, start:end]
        samples.append({
            'input_ids': input_ids,
            'attention_mask': torch.ones_like(input_ids),
        })

    torch.save(samples, cache_path)
    logger.info('Saved calibration cache: %s', cache_path)
    return samples


def quantize_variant_gptq(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantized_already = (ARTIFACT_DIR / 'config.json').exists() and find_gptq_model_basename(ARTIFACT_DIR) is not None
    if quantized_already:
        logger.info('Quantized artifact already exists at %s', ARTIFACT_DIR)
        zip_path = BUNDLE_DIR / f'{VARIANT}_quantized_model.zip'
        if not zip_path.exists():
            zip_directory(ARTIFACT_DIR, zip_path)
        return {
            'artifact_dir': str(ARTIFACT_DIR),
            'zip_path': str(zip_path),
            'gptq_model_file': find_gptq_model_basename(ARTIFACT_DIR),
            'dynamic_config': build_dynamic_config(VARIANT),
            'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
            'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        }

    samples = get_calibration_data(cfg, tokenizer)
    cal_dataset = [
        {
            'input_ids': s['input_ids'].squeeze(0),
            'attention_mask': s['attention_mask'].squeeze(0),
        }
        for s in samples
    ]

    qcfg_kwargs = {
        'bits': cfg['quant']['bits'],
        'group_size': cfg['quant']['group_size'],
        'desc_act': cfg['quant'].get('desc_act', False),
    }
    dynamic = build_dynamic_config(VARIANT)
    if dynamic is not None:
        qcfg_kwargs['dynamic'] = dynamic

    quantize_config = QuantizeConfig(**qcfg_kwargs)

    logger.info('Starting GPTQ %s for %s', VARIANT, cfg['base_model'])
    t0 = time.time()
    model = GPTQModel.load(cfg['base_model'], quantize_config, trust_remote_code=True)
    model.quantize(cal_dataset)
    model.save(str(ARTIFACT_DIR))
    tokenizer.save_pretrained(str(ARTIFACT_DIR))
    elapsed = time.time() - t0
    logger.info('Quantization finished in %s', format_seconds(elapsed))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_file = find_gptq_model_basename(ARTIFACT_DIR)
    if model_file is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {ARTIFACT_DIR}')

    manifest = {
        'variant': VARIANT,
        'base_model': cfg['base_model'],
        'quant': cfg['quant'],
        'dynamic_config': dynamic,
        'artifact_dir': str(ARTIFACT_DIR),
        'gptq_model_file': model_file,
        'quantization_elapsed_sec': elapsed,
        'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
        'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        'created_at_unix': time.time(),
    }
    save_json(manifest, ARTIFACT_DIR / 'artifact_manifest.json')
    save_json(manifest, RESULTS_DIR / 'quantization_manifest.json')

    zip_path = zip_directory(ARTIFACT_DIR, BUNDLE_DIR / f'{VARIANT}_quantized_model.zip')

    state = get_results_state()
    state['quantization'] = manifest | {'zip_path': str(zip_path)}
    save_results_state(state)
    summary = update_summary_table(state)

    print('Quantized artifact ready.')
    print(json.dumps(manifest, indent=2))
    display(pd.DataFrame([summary]))
    return manifest | {'zip_path': str(zip_path)}

Working directory: /content
Artifact dir: /content/artifacts/gptq_w4_g128/mlp_only_quant
Results dir: /content/results/gptq_w4_g128_mlp_only_quant/mlp_only_quant
Bundle dir: /content/zip_bundles/gptq_w4_g128_mlp_only_quant/mlp_only_quant
CUDA available: True
GPU: NVIDIA L4


In [ ]:
# Cell 1 — Quantize the model and save the quantized artifact zip
quant_manifest = quantize_variant_gptq(cfg)


2026-03-10 04:00:28,626 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:00:28,633 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
2026-03-10 04:00:28,642 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

2026-03-10 04:00:28,892 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:00:28,900 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-10 04:00:28,907 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-03-10 04:00:29,161 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-10 04:00:29,395 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-10 04:00:29,628 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:00:29,630 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-10 04:00:29,637 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/vocab.json "HTTP/1.1 200 OK"
2026-03-10 04:00:29,644 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848ed

...

INFO  | Process finalize   | 2     | 0.000  | 0.000  | 0.001   | 0.0%   | tp-pre-pad                                        |


INFO  +--------------------+-------+--------+--------+---------+--------+---------------------------------------------------+


2026-03-10 04:14:32,349 | INFO    | Quantization finished in 13:38
2026-03-10 04:16:27,992 | INFO    | Created zip: /content/zip_bundles/gptq_w4_g128_mlp_only_quant/mlp_only_quant/mlp_only_quant_quantized_model.zip (1079.38 MB)


Quantized artifact ready.
{
  "variant": "mlp_only_quant",
  "base_model": "Qwen/Qwen2-1.5B",
  "quant": {
    "bits": 4,
    "group_size": 128,
    "desc_act": false
  },
  "dynamic_config": {
    "-:.*self_attn\\.(q|k|v|o)_proj.*": {}
  },
  "artifact_dir": "/content/artifacts/gptq_w4_g128/mlp_only_quant",
  "gptq_model_file": "model.safetensors",
  "quantization_elapsed_sec": 818.4876461029053,
  "model_size_gb": 1.2827,
  "bits_per_param": 6.6632,
  "created_at_unix": 1773116072.4020019
}


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,mlp_only_quant,GPTQ W4 g128,1.2827,None,6.6632,None,None,None


In [ ]:
def evaluate_perplexity_only(model, tokenizer, cfg):
    ppl_cfg = cfg['eval']['perplexity']
    logger.info(
        'Loading Perplexity dataset: %s / %s / %s | max_length=%d stride=%d max_eval_tokens=%d',
        ppl_cfg['dataset'],
        ppl_cfg['dataset_name'],
        ppl_cfg['split'],
        ppl_cfg['max_length'],
        ppl_cfg['stride'],
        ppl_cfg.get('max_eval_tokens', 131072),
    )
    ds = load_dataset(ppl_cfg['dataset'], ppl_cfg['dataset_name'], split=ppl_cfg['split'])
    text = '\n\n'.join(t for t in ds['text'] if t and t.strip())
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = min(encodings.input_ids.size(1), ppl_cfg.get('max_eval_tokens', 131072))
    max_len = ppl_cfg['max_length']
    stride = ppl_cfg['stride']
    log_every = ppl_cfg.get('log_every', 25)

    nlls = []
    prev_end = 0
    start_time = time.time()
    peak_vram = 0.0
    windows = list(range(0, seq_len, stride))
    total_windows = max(1, len(windows))
    device = get_model_device(model)

    for idx, begin in enumerate(tqdm(windows, desc='Perplexity', unit='win'), start=1):
        end = min(begin + max_len, seq_len)
        input_ids = encodings.input_ids[:, begin:end].to(device)
        attention_mask = torch.ones_like(input_ids)
        target_ids = input_ids.clone()
        target_ids[:, :max(0, prev_end - begin)] = -100

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
            nlls.append(outputs.loss.detach().float())

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        prev_end = end
        if idx % log_every == 0 or end >= seq_len:
            elapsed = time.time() - start_time
            eta = (elapsed / idx) * (total_windows - idx) if idx else 0.0
            logger.info(
                'Perplexity progress %d/%d | elapsed=%s | eta=%s',
                idx,
                total_windows,
                format_seconds(elapsed),
                format_seconds(eta),
            )
        if end >= seq_len:
            break

    perplexity = math.exp(torch.stack(nlls).mean().item())
    elapsed = time.time() - start_time
    result = {
        'dataset': f"{ppl_cfg['dataset']}/{ppl_cfg['dataset_name']}",
        'split': ppl_cfg['split'],
        'max_length': max_len,
        'stride': stride,
        'num_tokens': int(seq_len),
        'num_windows': len(nlls),
        'perplexity': perplexity,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    save_json(result, RESULTS_DIR / 'perplexity_metrics.json')

    state = get_results_state()
    state['perplexity'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    logger.info(
        'Perplexity finished | ppl=%.4f | elapsed=%s | peak_vram_gb=%s',
        result['perplexity'],
        result['elapsed_hms'],
        result['peak_vram_gb'],
    )
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    return result


GSM_PROMPT = "Question: {question} Answer: Let's think step by step. "


def evaluate_gsm8k_only(model, tokenizer, cfg):
    gsm_cfg = cfg['eval']['gsm8k']
    n_shot = gsm_cfg['num_fewshot']
    num_samples = gsm_cfg['num_samples']
    max_new = gsm_cfg['max_new_tokens']
    log_every = gsm_cfg.get('log_every', 25)

    logger.info('Loading GSM8K: %d-shot, %d samples, max_new_tokens=%d', n_shot, num_samples, max_new)
    train_ds = load_dataset('openai/gsm8k', 'main', split='train').shuffle(seed=42)
    test_ds = load_dataset('openai/gsm8k', 'main', split='test').shuffle(seed=42).select(range(num_samples))
    exemplars = train_ds.select(range(n_shot))

    prefix = ''
    for ex in exemplars:
        prefix += GSM_PROMPT.format(question=ex['question']) + ex['answer'] + '\n\n'

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(test_ds)

    for idx, row in enumerate(tqdm(test_ds, desc='GSM8K', unit='q'), start=1):
        prompt = prefix + GSM_PROMPT.format(question=row['question'])

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new)
        pred = extract_numeric_answer(generated)
        gold = extract_numeric_answer(row['answer'])
        is_correct = pred is not None and gold is not None and abs(pred - gold) < 1e-3
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_value': pred,
            'gold_value': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('GSM8K', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'gsm8k',
        'num_fewshot': n_shot,
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'gsm8k_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'gsm8k_metrics.json')

    state = get_results_state()
    state['gsm8k'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    logger.info('GSM8K finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    return result


def format_arc_prompt(row):
    labels = row['choices']['label']
    texts = row['choices']['text']
    prompt = row['question'] + '\n'
    for lbl, txt in zip(labels, texts):
        prompt += f'{lbl}. {txt}\n'
    prompt += 'Answer:'
    return prompt


def evaluate_arc_only(model, tokenizer, cfg):
    arc_cfg = cfg['eval']['arc_challenge']
    num_samples = arc_cfg['num_samples']
    max_new = arc_cfg['max_new_tokens']
    log_every = arc_cfg.get('log_every', 50)

    logger.info('Loading ARC-Challenge: %d samples, max_new_tokens=%d', num_samples, max_new)
    ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test').shuffle(seed=42).select(range(num_samples))

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(ds)

    for idx, row in enumerate(tqdm(ds, desc='ARC-Challenge', unit='q'), start=1):
        prompt = format_arc_prompt(row)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new).strip()
        pred = generated[:1].upper() if generated else None
        gold = row['answerKey'].strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_label': pred,
            'gold_label': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('ARC-Challenge', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'arc_challenge',
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'arc_challenge_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'arc_challenge_metrics.json')

    state = get_results_state()
    state['arc_challenge'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    results_zip = zip_directory(RESULTS_DIR, BUNDLE_DIR / 'full_quant_results_only.zip')
    state['results_zip'] = str(results_zip)
    save_results_state(state)

    logger.info('ARC-Challenge finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    print('Results zip:', results_zip)
    return result

In [ ]:
# Cell 2 — Evaluate Perplexity only
# Run this after the quantization cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    perplexity_result = evaluate_perplexity_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


`torch_dtype` is deprecated! Use `dtype` instead!


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/wtwpfhyg-qaghghla/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


...

2026-03-10 04:21:25,923 | INFO    | Loading Perplexity dataset: wikitext / wikitext-2-raw-v1 / test | max_length=2048 stride=1024 max_eval_tokens=131072
2026-03-10 04:21:26,161 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:21:26,387 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:21:26,393 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
2026-03-10 04:21:26,627 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
2026-03-10 04:21:26,870 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71

Perplexity:   0%|          | 0/128 [00:00<?, ?win/s]

2026-03-10 04:21:37,620 | INFO    | Perplexity progress 25/128 | elapsed=0:07 | eta=0:31
2026-03-10 04:21:42,459 | INFO    | Perplexity progress 50/128 | elapsed=0:12 | eta=0:19
2026-03-10 04:21:47,325 | INFO    | Perplexity progress 75/128 | elapsed=0:17 | eta=0:12
2026-03-10 04:21:52,206 | INFO    | Perplexity progress 100/128 | elapsed=0:22 | eta=0:06
2026-03-10 04:21:57,097 | INFO    | Perplexity progress 125/128 | elapsed=0:27 | eta=0:00
2026-03-10 04:21:57,489 | INFO    | Perplexity progress 127/128 | elapsed=0:27 | eta=0:00
2026-03-10 04:21:57,631 | INFO    | Perplexity finished | ppl=8.5767 | elapsed=0:27 | peak_vram_gb=7.97538423538208


,dataset,split,max_length,stride,num_tokens,num_windows,perplexity,elapsed_sec,elapsed_hms,peak_vram_gb
0,wikitext/wikitext-2-raw-v1,test,2048,1024,131072,127,8.576692,27.546925,0:27,7.975384


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,mlp_only_quant,GPTQ W4 g128,1.2827,7.9754,6.6632,8.576692,None,None


In [ ]:

# Cell 3 — Evaluate on GSM8K only
# Run this after the Perplexity cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    gsm8k_result = evaluate_gsm8k_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:

# Cell 4 — Evaluate on ARC-Challenge only and finalize the summary/results zip
# Run this after the GSM8K cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    arc_result = evaluate_arc_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/pnnvijib-iqbxrhpb/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  The layer model.layers.0.self_attn.q_proj is not quantized.              


...

ARC-Challenge:   0%|          | 0/500 [00:00<?, ?q/s]

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: must be real number, not str
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    a

,dataset,num_samples,max_new_tokens,accuracy,correct,elapsed_sec,elapsed_hms,peak_vram_gb
0,arc_challenge,500,5,0.648,324,131.28405,2:11,5.705068


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,mlp_only_quant,GPTQ W4 g128,1.2827,7.9754,6.6632,8.576692,None,0.648


Results zip: /content/zip_bundles/gptq_w4_g128_mlp_only_quant/mlp_only_quant/full_quant_results_only.zip


In [15]:
state = get_results_state()
summary = update_summary_table(state)
print(json.dumps(state, indent=2))
display(pd.DataFrame([summary]))
print('''
Saved files under results:''')
for p in sorted(RESULTS_DIR.glob('*')):
    print('-', p.name)
print('''
Saved bundles:''')
for p in sorted(BUNDLE_DIR.glob('*')):
    print('-', p.name)

{
  "quantization": {
    "variant": "mlp_only_quant",
    "base_model": "Qwen/Qwen2-1.5B",
    "quant": {
      "bits": 4,
      "group_size": 128,
      "desc_act": false
    },
    "dynamic_config": {
      "-:.*self_attn\\.(q|k|v|o)_proj.*": {}
    },
    "artifact_dir": "/content/artifacts/gptq_w4_g128/mlp_only_quant",
    "gptq_model_file": "model.safetensors",
    "quantization_elapsed_sec": 818.4876461029053,
    "model_size_gb": 1.2827,
    "bits_per_param": 6.6632,
    "created_at_unix": 1773116072.4020019,
    "zip_path": "/content/zip_bundles/gptq_w4_g128_mlp_only_quant/mlp_only_quant/mlp_only_quant_quantized_model.zip"
  },
  "perplexity": {
    "dataset": "wikitext/wikitext-2-raw-v1",
    "split": "test",
    "max_length": 2048,
    "stride": 1024,
    "num_tokens": 131072,
    "num_windows": 127,
    "perplexity": 8.576692317577685,
    "elapsed_sec": 27.54692506790161,
    "elapsed_hms": "0:27",
    "peak_vram_gb": 7.97538423538208
  },
  "arc_challenge": {
    "dataset

,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,mlp_only_quant,GPTQ W4 g128,1.2827,7.9754,6.6632,8.576692,None,0.648



Saved files under results:
- arc_challenge_metrics.json
- arc_challenge_predictions.csv
- perplexity_metrics.json
- quantization_manifest.json
- state.json
- summary.csv
- summary.json

Saved bundles:
- full_quant_results_only.zip
- mlp_only_quant_quantized_model.zip
